# AtlasRank M4 Colab pilot

Preparation only. Read `docs/COLAB.md` and `docs/reviews/M4.colab.md` locally first.
Run the pilot only after the lead approves its exact manifest and private persistent destination.
Full catalog and training are excluded. All cells are unexecuted.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Operator fills these from the reviewed local preparation and approval.
INPUT_ZIP = Path("/content/atlasrank-colab-input-v5.zip")
INPUT_ZIP_SHA256 = "0c4019a93b3f12044e8d3f41fbbe4428eb7466cc0236e68ebe7c56bcbef873cf"
APPROVED_MANIFEST = Path("/content/m4-colab-pilot.retry-approved.json")
PERSISTENT = Path("/content/drive/MyDrive/AtlasRank/m4/pilot-l4-v1")
OWNER_TOKEN = "REPLACE_WITH_APPROVED_EXCLUSIVE_OWNER_TOKEN"
INPUTS = Path("/content/atlasrank-input")

Upload the private reviewed ZIP through Colab Files. The ZIP contains a source snapshot,
model files and label-free CPU pilot references. No GitHub repository is required.
The package checks the archive hash and extraction limits.


In [ ]:
# Load only the standard-library transport helper from the reviewed ZIP.
sys.path.insert(0, str(INPUT_ZIP) + "/source/src")
from atlasrank.retrieval.colab_transport import extract_inputs

extract_inputs(INPUT_ZIP, INPUTS, INPUT_ZIP_SHA256)
os.environ["PYTHONPATH"] = str(INPUTS / "source/src")

Discovery reads the actual native Torch build and allocated GPU. The reported NVIDIA-SMI
CUDA version alone doesn't establish Torch compatibility. Export this output if preflight
rejects the environment. Discovery performs no embedding inference.


In [ ]:
subprocess.run(
    [sys.executable, "-m", "atlasrank.retrieval.colab_environment", "inspect"],
    check=True,
)

Mount only your approved private Drive account. Confirm the exact directory and that
no other AtlasRank runtime owns this job. The pilot runner requires matching explicit
approval and owner token; it preserves the native CUDA Torch build.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# STOP until the lead has approved this file, configuration and destination.
os.environ["ATLAS_COLAB_EXCLUSIVE_RUNTIME"] = OWNER_TOKEN
subprocess.run(
    [
        sys.executable,
        "-m",
        "atlasrank.retrieval.colab_runner",
        "--manifest",
        str(APPROVED_MANIFEST),
        "--inputs",
        str(INPUTS),
        "--persistent",
        str(PERSISTENT),
        "--work",
        "/content/atlasrank-work",
    ],
    check=True,
)

The package writes receipts, completed chunks, attempt charges, environment and timing
evidence to the approved folder. A passing command isn't confirmation of Drive cloud
durability. Stop this runtime, download the complete folder, and perform the documented
fresh-runtime restore and local FAISS review before approving any full run.
Don't remove failed uploads or prior attempts. Don't rerun to reset a time cap.
